<a href="https://colab.research.google.com/github/NikoriakViktor/PY-Course-Victor-Nikoriak-22-09-2026/blob/main/module_2/lessons/lesson_24_property_decorators_dunder/note_lesson_24_property_dunder.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Урок 24 — `@property`, декоратори класів, dunder

Мобільний застосунок сервісу «Смачно + Таксі» пише звичайний Python поверх наших класів: `sorted(deliveries)`, `len(cart)`, `"Борщ" in cart`, `price + fee`. Сьогодні вчимо класи працювати з цим синтаксисом:

1. **Dunder-методи** — протоколи Python: контейнер, арифметика, порівняння, хеш, виклик.
2. **`@property` і дескриптори** — обчислювані атрибути й одна перевірка на багато полів.
3. **Декоратори класів** — реєстр, `@total_ordering`, `@dataclass`.

| Розділ | Що всередині | Приклади | Задача |
|---|---|---|---|
| 1 | протоколи Python і **контейнер**: `__len__`, `__contains__`, `__iter__`, `__getitem__`, `__bool__` | 1.1–1.6 | **Задача 1** |
| 2 | **гроші**: `__add__`, `NotImplemented`, `__repr__` і `__str__`, `__radd__` і `sum`, `__mul__`, `+=` | 2.1–2.8 | |
| 3 | **рівність, хеш, порядок, виклик**: `__eq__` + `__hash__`, `__lt__` чи `key=`, `__call__` | 3.1–3.8 | |
| 4 | **`@property`**: обчислювані атрибути, рекурсія в сеттері, `property()` без декоратора, `cached_property` | 4.1–4.8 | |
| 5 | **дескриптори**: `__set_name__`, `__get__`, `__set__`, порядок пошуку атрибута | 5.1–5.5 | **Задача 2** |
| 6 | **декоратори класів**: реєстр, `@total_ordering`, `@dataclass`, `__post_init__`, `default_factory` | 6.1–6.10 | **Задача 3** |
| 7 | значення й сутність; розібраний приклад «кошик із грошима» | 7.1–7.3 | |

**Як працювати**

- Виконуй клітинки **зверху вниз**. Клітинка з визначенням (класи, функції, константи) лише створює їх — **запуск у наступній клітинці**. Так приклад можна перезапускати, не перевизначаючи класів.
- Перед **🔮 Прогнозом** спершу скажи, що буде, потім розгортай відповідь.
- Класи нарощуємо частинами: `class Money(Money)` бере попередню версію й додає метод. У справжньому проєкті всі методи живуть в одному класі.
- Три **задачі** розв'язуєш сам: код — у клітинці задачі, перевірка — у наступній.
- Теорія, схеми й архітектура — у книзі: [Урок 24. @property, декоратори класів, dunder](https://nikoriakviktor.github.io/PY-Course-Victor-Nikoriak-22-09-2026/modules/m2/lesson_24/).

## 🔁 Пригадай (без підглядання)

1. Що приймає і що повертає декоратор функції (урок 10)?
2. Яке правило пов'язує хеш і рівність ключів словника (урок 17)?
3. Що робить `@property` без сеттера (урок 22)?

<details>
<summary>Відповіді</summary>

1. Приймає функцію й повертає функцію.
2. Рівні ключі мусять мати однаковий хеш.
3. Дає читати метод як атрибут; запис падає з `AttributeError`.

</details>

---
## 1. Dunder-методи: протоколи Python

Коли Python бачить `len(cart)`, він не шукає функцію `len` усередині кошика, а викликає **спеціальний метод** класу: `type(cart).__len__(cart)`. Методи з двома підкресленнями з обох боків — **dunder** (double underscore). Набір таких методів — **протокол**: реалізував потрібні — і об'єкт працює з вбудованим синтаксисом, як список чи число.

| Вираз | Що викликає Python | Протокол |
|---|---|---|
| `len(x)` | `x.__len__()` | розмір |
| `item in x` | `x.__contains__(item)` | належність |
| `for item in x` | `x.__iter__()` | ітерація |
| `x[key]` | `x.__getitem__(key)` | доступ за ключем |
| `bool(x)`, `if x:` | `x.__bool__()`, інакше `x.__len__()` | істинність |
| `a + b` | `a.__add__(b)`, інакше `b.__radd__(a)` | арифметика |
| `a == b`, `a < b` | `a.__eq__(b)`, `a.__lt__(b)` | порівняння |
| `hash(x)` | `x.__hash__()` | хешування |
| `repr(x)`, `str(x)` | `x.__repr__()`, `x.__str__()` | подання |
| `x(arg)` | `x.__call__(arg)` | виклик |

### Приклад 1.1. Клас без протоколів

Задача: відсортувати доставки й порівняти дві однакові. Клас поки знає лише свої поля.

🔮 **Прогноз:** що буде з `sorted(deliveries)` і чи рівні дві однакові доставки?

In [ ]:
class Delivery:
    def __init__(self, order_id, minutes):
        self.order_id = order_id
        self.minutes = minutes

In [ ]:
deliveries = [Delivery(1, 35), Delivery(2, 20), Delivery(3, 50)]
try:
    sorted(deliveries)
except TypeError as error:
    print(error)
print(Delivery(1, 35) == Delivery(1, 35))

<details>
<summary>Відповідь</summary>

`TypeError`: Python не знає, що означає «менша доставка». А `==` за замовчуванням порівнює **ідентичність** — це два різні об'єкти, тому `False`.

</details>

### Приклад 1.2. Шпигун: які методи викликає Python

Щоб побачити протоколи «зсередини», напишемо клас, кожен dunder-метод якого друкує, що його викликали. Нічого корисного він не робить — лише показує, **хто** працює за кожним виразом.

In [ ]:
class Spy:
    def __len__(self):
        print("  → __len__")
        return 3

    def __contains__(self, item):
        print(f"  → __contains__({item!r})")
        return item == "Борщ"

    def __iter__(self):
        print("  → __iter__")
        return iter(["Борщ", "Узвар"])

    def __getitem__(self, key):
        print(f"  → __getitem__({key!r})")
        return 95

In [ ]:
spy = Spy()
print("len:", len(spy))
print("in:", "Борщ" in spy)
print("list:", list(spy))
print("[]:", spy["Борщ"])
print("напряму:", type(spy).__len__(spy))

spy.__len__ = lambda: 100          # метод в ЕКЗЕМПЛЯРІ len() ігнорує
print("після підміни:", len(spy))

Рядок `→ …` друкується **перед** результатом: спершу Python викликає метод, потім `print` показує, що той повернув. `list(spy)` викликав `__iter__`, а ще й `__len__` — `list` заздалегідь питає розмір, щоб виділити пам'ять. `len(spy)` і `type(spy).__len__(spy)` — одне й те саме. А підміна `spy.__len__` в екземплярі нічого не змінила: вбудовані операції шукають dunder-методи **в класі**, а не в об'єкті.

**Правило:** вираз Python → dunder-метод класу. Хочеш, щоб твій об'єкт підтримував `len`, `in`, `for` — напиши відповідний метод у класі.

### Приклад 1.3. Кошик як контейнер

Спершу — **картка класу**: що це за річ і що вона вміє. Кожен рядок картки знайдеш у коді.

| | `Cart` |
|---|---|
| **Що це?** | кошик одного замовлення |
| **Знає** | `_items` — словник «страва → кількість» (внутрішній, з `_`) |
| **Вміє** | `add(dish, qty)`; `len(cart)` — кількість порцій; `dish in cart`; `cart[dish]`; `for dish, qty in cart`; `repr` |
| **Правила** | однакова страва не дублюється — збільшується кількість |
| **Не робить** | не знає цін і не рахує суму (з'явиться в розділі 4) |

In [ ]:
class Cart:
    def __init__(self):
        self._items = {}

    def add(self, dish, qty=1):
        self._items[dish] = self._items.get(dish, 0) + qty

    def __len__(self):
        return sum(self._items.values())

    def __contains__(self, dish):
        return dish in self._items

    def __iter__(self):
        return iter(self._items.items())

    def __getitem__(self, dish):
        return self._items[dish]

    def __repr__(self):
        return f"Cart({self._items})"

In [ ]:
cart = Cart()
cart.add("Борщ", 2)
cart.add("Узвар")
print(len(cart), "Борщ" in cart, cart["Борщ"])
for dish, qty in cart:
    print(dish, qty)
print(cart)

`len(cart)` дає 3, бо рахує **порції** (2 борщі + 1 узвар), а не рядки, — це наше рішення, і його видно в одному методі `__len__`. `_items` лишився внутрішнім, а зовнішній код користується кошиком як звичайною колекцією.

### Приклад 1.4. Істинність без `__bool__`

🔮 **Прогноз:** методу `__bool__` у кошика немає. Що надрукує `bool(Cart())` і `bool(cart)`?

In [ ]:
print(bool(Cart()), bool(cart))
if cart:
    print("у кошику щось є")

<details>
<summary>Відповідь</summary>

`False True`: без `__bool__` Python бере `__len__`, нульова довжина — хибність. Тому `if cart:` читається природно — «якщо в кошику щось є».

</details>

### Приклад 1.5. `__bool__` важливіший за `__len__`

Задача: зміна кур'єра може мати замовлення, але вже бути **закритою**. «Чи можна працювати зі зміною» — це не «чи є в ній замовлення». Додаємо `__bool__` і дивимось, хто з методів викликається.

In [ ]:
class Shift:
    def __init__(self, orders, is_open):
        self.orders = orders
        self.is_open = is_open

    def __len__(self):
        print("  → __len__")
        return len(self.orders)

    def __bool__(self):
        print("  → __bool__")
        return self.is_open

In [ ]:
closed = Shift(orders=[101, 102], is_open=False)
print("bool:", bool(closed))
print("len:", len(closed))

Для `bool(closed)` викликався лише `__bool__` і повернув `False`, хоч замовлень два. `__len__` спрацював тільки для `len()`.

**Правило:** `if x:` спершу шукає `__bool__`, потім `__len__`; якщо немає жодного — об'єкт завжди істинний.

### Приклад 1.6. Лише `__getitem__` — і вже працюють `for` та `in`

Меню дня — проста послідовність страв за номером. Напишемо **тільки** `__getitem__(index)` і подивимось, що вміє такий об'єкт.

In [ ]:
class DailyMenu:
    def __init__(self, dishes):
        self._dishes = list(dishes)

    def __getitem__(self, index):
        print(f"  → [{index}]")
        return self._dishes[index]

In [ ]:
today = DailyMenu(["Борщ", "Деруни", "Узвар"])
print(list(today))
print("Деруни" in today)
print(today[1:])

Без `__iter__` цикл `for` (і `list`) викликає `__getitem__(0)`, `__getitem__(1)`, … доки не отримає `IndexError` — тому в журналі є `[3]`, що вже за межами списку. `in` перебирає так само й зупиняється на першому збігу (`[0]`, `[1]`). Зріз `today[1:]` теж працює: `index` — це об'єкт `slice`, а список уміє з ним працювати.

Це старий «протокол послідовності». Для контейнера краще писати явні `__iter__` і `__contains__`: швидше і зрозуміліше.

### Задача 1. Меню як контейнер

Картка класу вже заповнена — переклади її в код:

| | `Menu` |
|---|---|
| **Що це?** | меню ресторану: страва → ціна |
| **Знає** | `_prices` — копія словника цін |
| **Вміє** | `len(menu)` — кількість страв; `dish in menu`; `menu[dish]` — ціна; `for dish in menu` — назви **від найдешевшої**; `repr` → `Menu(страв: 4)` |
| **Правила** | порожнє меню хибне (`not Menu({})`) — без окремого `__bool__` |
| **Не робить** | не змінює ціни й не знає про кошик |

Перевірка — у наступній клітинці.

In [ ]:
class Menu:
    def __init__(self, prices):
        self._prices = dict(prices)

    # BEGIN SOLUTION
    def __len__(self):
        return len(self._prices)

    def __contains__(self, dish):
        return dish in self._prices

    def __getitem__(self, dish):
        return self._prices[dish]

    def __iter__(self):
        return iter(sorted(self._prices, key=self._prices.get))

    def __repr__(self):
        return f"Menu(страв: {len(self)})"
    # END SOLUTION

In [ ]:
menu = Menu({"Борщ": 95, "Вареники": 110, "Узвар": 40, "Хліб": 10})
print(menu, list(menu))
assert len(menu) == 4 and "Узвар" in menu and "Піца" not in menu
assert menu["Борщ"] == 95
assert list(menu) == ["Хліб", "Узвар", "Борщ", "Вареники"]
assert repr(menu) == "Menu(страв: 4)"
assert not Menu({})
print("✅ Задача 1 розв'язана")

---
## 2. Гроші: арифметика й `NotImplemented`

Ціни як голі числа легко переплутати: хвилини, кілометри й гривні — усе `int`. Клас `Money` робить одиницю явною й дозволяє складати лише гроші з грошима.

| | `Money` |
|---|---|
| **Що це?** | сума в гривнях — **об'єкт-значення** |
| **Знає** | `amount` |
| **Вміє** | `+` з іншими грошима, `==` за сумою, `repr` для розробника, `str` для людини |
| **Правила** | гроші + число — помилка: невідомо, гривні це чи хвилини |
| **Не робить** | не знає, за що заплачено, і не конвертує валюти |

### Приклад 2.1. `__add__` і `__eq__`

In [ ]:
class Money:
    def __init__(self, amount):
        self.amount = amount

    def __add__(self, other):
        if not isinstance(other, Money):
            return NotImplemented
        return Money(self.amount + other.amount)

    def __eq__(self, other):
        if not isinstance(other, Money):
            return NotImplemented
        return self.amount == other.amount

    def __repr__(self):
        return f"Money({self.amount})"

    def __str__(self):
        return f"{self.amount} грн"

In [ ]:
bill = Money(95) + Money(60)
print(bill, repr(bill), bill == Money(155))
try:
    Money(95) + 60
except TypeError as error:
    print(error)

`Money(95) + Money(60)` дав **новий** об'єкт `Money(155)`; `==` порівнює суми. `Money(95) + 60` упав з `TypeError` — і це добре: число без одиниці до грошей не додаємо.

### Приклад 2.2. `__repr__` чи `__str__`: хто що бачить

`__repr__` — для розробника (однозначно, як створити об'єкт), `__str__` — для людини. Подивимось, де Python бере кожен.

In [ ]:
price = Money(95)
print(price)
print(str(price), "|", repr(price))
print(f"до сплати: {price}; у логах: {price!r}")
print([price, Money(40)])

`print` і f-рядок беруть `__str__` (`95 грн`), а `!r` — `__repr__`. Список друкує свої елементи через `repr`: `[Money(95), Money(40)]`. Тому `__repr__` пишуть завжди, а `__str__` — коли для людини потрібен інший вигляд. Без `__str__` Python використав би `__repr__`.

### Приклад 2.3. `NotImplemented` — не помилка, а сигнал

Викличемо `__add__` напряму — без оператора `+`.

In [ ]:
print(Money(95).__add__(Money(5)))
print(Money(95).__add__(60))
print(type(NotImplemented).__name__)

Метод сам **не кидає** помилку — він повертає спеціальне значення `NotImplemented`: «я не вмію з цим типом». Помилку кидає оператор `+`, і лише після того, як дав шанс іншому операнду:

```text
a + b
 │
 ├─ a.__add__(b) ──── результат? ──► готово
 │        │
 │   NotImplemented
 │        ▼
 ├─ b.__radd__(a) ─── результат? ──► готово
 │        │
 │   NotImplemented
 │        ▼
 └─ TypeError: unsupported operand type(s)
```

Якби `__add__` кидав `TypeError` сам, Python не спробував би `__radd__` іншого класу.

### Приклад 2.4. Пастка `sum`

🔮 **Прогноз:** чи спрацює `sum(prices)` для списку `Money`?

In [ ]:
prices = [Money(95), Money(60), Money(40)]
try:
    sum(prices)
except TypeError as error:
    print(error)
print(sum(prices, Money(0)))

<details>
<summary>Відповідь</summary>

Ні: `sum` починає з `0`, тобто перший крок — `0 + Money(95)`. `int` не вміє додавати гроші, а `Money.__radd__` немає. Зі стартовим значенням `Money(0)` усе працює: `Money(0) + Money(95)` — це наш `__add__`.

</details>

### Приклад 2.5. `__radd__`: «праве» додавання

Другий вихід — навчити гроші додаватися, коли вони стоять **праворуч** від числа. Підтримуємо лише нуль — саме з нього стартує `sum`.

In [ ]:
class Money(Money):
    def __radd__(self, other):
        if other == 0:
            return self
        return NotImplemented

In [ ]:
print(sum([Money(95), Money(60), Money(40)]))
print(0 + Money(10))
try:
    5 + Money(10)
except TypeError as error:
    print(error)

`0 + Money(10)`: `int.__add__` повернув `NotImplemented`, і Python викликав `Money(10).__radd__(0)` — він повернув самі гроші. `5 + Money(10)` і далі помилка: п'ять чого — гривень, хвилин?

### Приклад 2.6. `__mul__` і `__rmul__`: ціна × кількість

Два борщі — це `Money(95) * 2`. А хтось напише `2 * Money(95)` — для цього потрібне дзеркальне `__rmul__`.

In [ ]:
class Money(Money):
    def __mul__(self, times):
        if not isinstance(times, int):
            return NotImplemented
        return Money(self.amount * times)

    def __rmul__(self, times):
        return self * times

In [ ]:
print(Money(95) * 2, 2 * Money(95))
try:
    Money(95) * Money(2)
except TypeError as error:
    print(error)

Гроші × ціле число — гроші; обидва порядки дають `190 грн`. Гроші × гроші не мають сенсу («гривні в квадраті»), тому `NotImplemented` і `TypeError`.

### Приклад 2.7. `+=` без `__iadd__`

🔮 **Прогноз:** чи залишиться `price` тим самим об'єктом після `price += Money(5)`? А список після `items += [2]`?

In [ ]:
price = Money(95)
same_price = price
price += Money(5)
print(price, same_price, price is same_price)

items = [1]
same_items = items
items += [2]
print(items, same_items, items is same_items)

<details>
<summary>Відповідь</summary>

`Money` не має `__iadd__`, тож `price += Money(5)` означає `price = price + Money(5)` — **новий** об'єкт; стара змінна `same_price` лишилась 95 грн. Список має `__iadd__`, який змінює його **на місці**: `same_items` теж став `[1, 2]`. Для об'єктів-значень так і треба — гроші не змінюються, а замінюються.

</details>

### Приклад 2.8. Знайди помилку: `__add__`, що змінює лівий операнд

Поширена помилка — «додати до себе» і повернути `self`.

In [ ]:
class LeakyMoney:
    def __init__(self, amount):
        self.amount = amount

    def __add__(self, other):
        self.amount += other.amount      # змінює лівий операнд!
        return self

    def __repr__(self):
        return f"LeakyMoney({self.amount})"

In [ ]:
price = LeakyMoney(95)
fee = LeakyMoney(30)
total = price + fee
print(total, price, total is price)

Після `total = price + fee` змінилася й сама ціна страви: `price` тепер 125, а `total` — той самий об'єкт. Наступне замовлення отримає зіпсовану ціну.

**Правило:** арифметичний оператор повертає **новий** об'єкт і не чіпає операнди: `return Money(self.amount + other.amount)`.

---
## 3. Рівність, хеш, порядок, виклик

### Приклад 3.1. `__eq__` забирає `__hash__`

🔮 **Прогноз:** чи можна покласти два `Money(95)` у множину?

In [ ]:
try:
    {Money(95), Money(95)}
except TypeError as error:
    print(error)
print(Money.__hash__)

<details>
<summary>Відповідь</summary>

Ні: визначивши `__eq__`, клас втратив `__hash__` (він став `None`). Правило з уроку 17: рівні об'єкти мусять мати однаковий хеш. Стандартний хеш рахується з ідентичності, тож два рівні `Money(95)` мали б різні хеші — Python волів відмовити, ніж тихо помилятися.

</details>

### Приклад 3.2. Хеш із тих самих полів, що й рівність

In [ ]:
class Money(Money):
    def __hash__(self):
        return hash(self.amount)

In [ ]:
print(len({Money(95), Money(95), Money(60)}), Money(95) in {Money(95)})
print({Money(95): "борщ"}[Money(95)])

Тепер два `Money(95)` — один елемент множини, а гроші можна брати ключем словника: `Money(95)`, створений заново, знаходить запис.

### Приклад 3.3. Монета, що загубилась у множині

🔮 **Прогноз:** монету змінили після того, як поклали в множину. Чи знайде її `in`?

In [ ]:
wallet = {Money(95)}
coin = next(iter(wallet))
coin.amount = 100
print(Money(100) in wallet, coin in wallet, len(wallet))

<details>
<summary>Відповідь</summary>

`False False 1`: монета в множині є, але лежить у комірці **старого** хешу (від 95). Шукаючи `Money(100)`, множина дивиться в іншу комірку, а шукаючи за старим хешем — знаходить об'єкт, що вже не дорівнює 95. Хешованим має бути лише **незмінний** об'єкт (як це зробити одним рядком — у розділі 6).

</details>

### Приклад 3.4. Знайди помилку: `__eq__` без перевірки типу

In [ ]:
class RawMoney:
    def __init__(self, amount):
        self.amount = amount

    def __eq__(self, other):
        return self.amount == other.amount

In [ ]:
try:
    print(RawMoney(5) == 5)
except AttributeError as error:
    print("RawMoney:", error)
print("Money:", Money(5) == 5)

`RawMoney(5) == 5` падає: в числа немає `amount`. Наш `Money` повертає `NotImplemented`, Python пробує `5 == Money(5)` з боку `int`, теж отримує `NotImplemented` — і чесно відповідає `False`. Для `==` (на відміну від `+`) Python наприкінці не кидає помилку, а порівнює ідентичність.

### Приклад 3.5. Порівняння й сортування: досить `__lt__`

🔮 **Прогноз:** у класі є лише `__lt__`. Що буде з `sorted`, `max` і `<=`?

In [ ]:
class Delivery:
    def __init__(self, order_id, minutes):
        self.order_id = order_id
        self.minutes = minutes

    def __lt__(self, other):
        return self.minutes < other.minutes

    def __repr__(self):
        return f"Delivery(№{self.order_id}, {self.minutes} хв)"

In [ ]:
deliveries = [Delivery(1, 35), Delivery(2, 20), Delivery(3, 50)]
print(sorted(deliveries))
print(max(deliveries))
try:
    Delivery(1, 35) <= Delivery(2, 20)
except TypeError as error:
    print(error)

<details>
<summary>Відповідь</summary>

`sorted` і `max` працюють (`a > b` Python перевертає в `b < a`), а для `<=` пари немає — `TypeError`. Решту порівнянь допише `@total_ordering` (розділ 6).

</details>

### Приклад 3.6. Як Python перевертає `>` у `<`

Додамо друк у `__lt__`, щоб побачити, **чий** метод викликається.

In [ ]:
class LoudDelivery:
    def __init__(self, name, minutes):
        self.name = name
        self.minutes = minutes

    def __lt__(self, other):
        print(f"  → {self.name}.__lt__({other.name})")
        return self.minutes < other.minutes

In [ ]:
fast, slow = LoudDelivery("швидка", 20), LoudDelivery("повільна", 50)
print("fast < slow:", fast < slow)
print("fast > slow:", fast > slow)

Для `fast > slow` методу `__gt__` немає, тому Python викликав **дзеркальний** метод правого операнда: `slow.__lt__(fast)`. Для `<` і `>` дзеркальні пари є, для `<=` — це `>=`, якого теж немає.

### Приклад 3.7. `__lt__` чи `key=`

Яка доставка «менша»? Швидша? Дешевша? Раніша за номером? Природного порядку в доставок немає, тому надійніше передати порядок явно, як в уроці 19.

In [ ]:
print(sorted(deliveries, key=lambda delivery: delivery.order_id, reverse=True))
print(min(deliveries, key=lambda delivery: delivery.order_id))

`key=` перемагає `__lt__`: сьогодні звіт сортує за номером, завтра — за часом, і клас для цього не змінюється.

**Правило:** `__lt__` — лише коли порядок один і очевидний (гроші, час, версії). Інакше — `key=`.

### Приклад 3.8. `__call__`: об'єкт як функція

У вечірні години тариф множиться на коефіцієнт. Коефіцієнт — налаштування, а застосувати його треба як функцію. Порівняємо замикання з уроку 19 і клас із `__call__`.

In [ ]:
def make_surge(factor):
    def apply(fare):
        return round(fare * factor)
    return apply


class Surge:
    def __init__(self, factor):
        self.factor = factor

    def __call__(self, fare):
        return round(fare * self.factor)

    def __repr__(self):
        return f"Surge(×{self.factor})"

In [ ]:
evening_fn = make_surge(1.5)
evening = Surge(1.5)
print(evening_fn(120), evening(120), callable(evening))
print(list(map(evening, [100, 80])))
print(evening)
evening.factor = 2
print(evening, evening(120))

Обидва варіанти викликаються однаково й годяться для `map`. Але об'єкт `Surge` має зрозумілий `repr` і налаштування, яке видно й можна змінити (`evening.factor = 2`). Замикання ховає `factor` усередині.

**Правило:** одне налаштування — замикання; кілька налаштувань, `repr` чи зміна на льоту — клас із `__call__`.

---
## 4. `@property` докладніше

В уроці 22 `@property` захищав запис. Друге його призначення — **обчислювані атрибути**: значення не зберігається, а рахується щоразу, коли його читають.

### Приклад 4.1. Сума кошика як властивість

In [ ]:
PRICES = {"Борщ": 95, "Вареники": 110, "Узвар": 40}


class Cart(Cart):
    @property
    def total(self):
        return sum(PRICES[dish] * qty for dish, qty in self)

    def remove(self, dish):
        del self._items[dish]

In [ ]:
cart = Cart()
cart.add("Борщ", 2)
cart.add("Узвар")
print(cart.total)
cart.add("Вареники")
print(cart.total)
cart.remove("Борщ")
print(cart.total, cart)

`cart.total` читається як поле, але щоразу рахується з `_items`: 230 → 340 → 150 після видалення борщу. `total` перебирає `self`, тобто користується нашим же `__iter__` з розділу 1.

### Приклад 4.2. Збережене поле застаріває

Що, якби замість властивості ми **зберігали** суму в полі й оновлювали її в `add`?

In [ ]:
class StoredCart:
    def __init__(self):
        self._items = {}
        self.total = 0

    def add(self, dish, qty=1):
        self._items[dish] = self._items.get(dish, 0) + qty
        self.total += PRICES[dish] * qty

    def remove(self, dish):
        del self._items[dish]            # забули оновити total

In [ ]:
stored = StoredCart()
stored.add("Борщ", 2)
stored.add("Узвар")
stored.remove("Борщ")
print(stored.total, stored._items)

У кошику лише узвар (40 грн), а `total` каже 230: новий метод `remove` забув оновити поле. Обчислювана властивість **не може застаріти** — у неї одне джерело правди, `_items`.

### Приклад 4.3. Пастка: сеттер, що викликає сам себе

🔮 **Прогноз:** що станеться при створенні `Courier`?

In [ ]:
class Courier:
    def __init__(self, name, rating):
        self.name = name
        self.rating = rating

    @property
    def rating(self):
        return self.rating

    @rating.setter
    def rating(self, value):
        if not 1 <= value <= 5:
            raise ValueError("рейтинг має бути від 1 до 5")
        self.rating = value

In [ ]:
try:
    Courier("Олег", 4.8)
except RecursionError as error:
    print(type(error).__name__)

<details>
<summary>Відповідь</summary>

`RecursionError`: `self.rating = value` у сеттері — знову присвоєння властивості, тобто знову виклик сеттера, і так до переповнення стеку (урок 23). Зберігати треба в іншому імені — `self._rating`.

</details>

### Приклад 4.4. Правильний сеттер

Значення лежить у `_rating`, а в `__init__` пишемо саме `self.rating = rating` — так перевірка спрацює й при створенні об'єкта.

In [ ]:
class Courier:
    def __init__(self, name, rating):
        self.name = name
        self.rating = rating             # через сеттер — з перевіркою

    @property
    def rating(self):
        return self._rating

    @rating.setter
    def rating(self, value):
        if not 1 <= value <= 5:
            raise ValueError(f"рейтинг має бути від 1 до 5, а маємо {value}")
        self._rating = value

In [ ]:
oleh = Courier("Олег", 4.8)
print(oleh.rating, vars(oleh))
for bad in [lambda: Courier("Ірина", 7), lambda: setattr(oleh, "rating", 0)]:
    try:
        bad()
    except ValueError as error:
        print(error)
print(oleh.rating)

У `vars(oleh)` значення лежить під `_rating`, а `rating` — властивість у класі. Поганий рейтинг не пройшов ні при створенні, ні при зміні, і в Олега лишилось 4.8.

### Приклад 4.5. Властивість лише для читання

Рівень кур'єра залежить від рейтингу — це обчислюване значення, записувати його не можна. Властивість без сеттера.

In [ ]:
class Courier(Courier):
    @property
    def level(self):
        return "золотий" if self.rating >= 4.8 else "звичайний"

In [ ]:
oleh = Courier("Олег", 4.8)
print(oleh.level)
oleh.rating = 4.5
print(oleh.level)
try:
    oleh.level = "золотий"
except AttributeError as error:
    print(type(error).__name__)

Рівень змінився сам, щойно змінився рейтинг. Записати `level` напряму не вийшло — `AttributeError`: «золотий» рівень не можна призначити, його можна лише заробити рейтингом.

### Приклад 4.6. `property()` без декоратора

`@property` — це звичайний виклик функції `property(геттер, сеттер)`. Запишемо чайові так, як `@property` робить за нас.

In [ ]:
class Tip:
    def __init__(self, percent):
        self.percent = percent

    def _get_percent(self):
        return self._percent

    def _set_percent(self, value):
        if not 0 <= value <= 30:
            raise ValueError(f"чайові від 0 до 30 %, а маємо {value}")
        self._percent = value

    percent = property(_get_percent, _set_percent)

In [ ]:
tip = Tip(10)
print(tip.percent, type(vars(Tip)["percent"]).__name__)
try:
    tip.percent = 50
except ValueError as error:
    print(error)

У класі `Tip` під іменем `percent` лежить об'єкт типу `property`, який знає дві функції. Синтаксис `@property` / `@percent.setter` будує той самий об'єкт. Об'єкт у класі, що перехоплює читання й запис атрибута, називають **дескриптором** — про них розділ 5.

### Приклад 4.7. `cached_property`: порахувати один раз

🔮 **Прогноз:** скільки разів надрукується «рахую маршрут…»?

In [ ]:
from functools import cached_property


class Route:
    def __init__(self, stops):
        self.stops = stops

    @cached_property
    def length(self):
        print("рахую маршрут…")
        return sum(abs(b - a) for a, b in zip(self.stops, self.stops[1:]))

In [ ]:
route = Route([0, 4, 1, 7])
print(route.length)
print(route.length)
print(vars(route))

<details>
<summary>Відповідь</summary>

Один: результат ліг в атрибут екземпляра — `vars(route)` тепер містить `'length': 13`, і наступне читання знаходить його там, не викликаючи функцію.

</details>

### Приклад 4.8. Ціна кешу — застарілість

Що буде, якщо після першого читання змінити зупинки?

In [ ]:
route = Route([0, 4, 1, 7])
print(route.length)
route.stops = [0, 10]
print("після зміни:", route.length)
del route.length                     # прибрати кеш з екземпляра
print("після del:", route.length)

Після зміни `stops` довжина лишилась старою — 13, а не 10. Лише `del route.length` прибрав збережене значення, і наступне читання порахувало заново.

**Правило:** `property` — рахує щоразу й не застаріває; `cached_property` — один раз, тому лише для даних, що не змінюються.

---
## 5. Дескриптори: одна перевірка на багато полів

Тариф таксі: подача, ціна кілометра й мінімальна вартість — усі мають бути більшими за нуль. Через `@property` це три геттери й три сеттери з однаковим `if value <= 0`. Повтор — сигнал винести правило в окремий об'єкт.

| | `Positive` |
|---|---|
| **Що це?** | правило «значення > 0» для одного поля класу — **дескриптор** |
| **Знає** | `name` — ім'я поля, під яким зберігати значення в екземплярі (`_per_km`) |
| **Вміє** | `__set_name__` — дізнатися ім'я поля; `__get__` — віддати значення; `__set__` — перевірити й записати |
| **Правила** | значення зберігає в **екземплярі**, а не в собі (один дескриптор на всі тарифи) |
| **Не робить** | не знає, що таке тариф і як рахується поїздка |

### Приклад 5.1. Дескриптор `Positive` і тариф

In [ ]:
class Positive:
    def __set_name__(self, owner, name):
        self.name = "_" + name

    def __get__(self, obj, objtype=None):
        if obj is None:
            return self
        return getattr(obj, self.name)

    def __set__(self, obj, value):
        if value <= 0:
            raise ValueError(f"{self.name[1:]} має бути більшим за 0, а маємо {value}")
        setattr(obj, self.name, value)


class Tariff:
    base = Positive()
    per_km = Positive()
    min_fare = Positive()

    def __init__(self, base, per_km, min_fare):
        self.base = base
        self.per_km = per_km
        self.min_fare = min_fare

    def fare(self, km):
        return max(self.min_fare, self.base + self.per_km * km)

In [ ]:
day = Tariff(40, 12, 80)
print(day.fare(2), day.fare(10))
try:
    day.per_km = -5
except ValueError as error:
    print(error)
print(vars(day))

Що відбулося:

1. Коли Python створював клас `Tariff`, він викликав `__set_name__` для кожного дескриптора — так `Positive()` дізнався, що його звуть `per_km`, і зберігатиме значення в `_per_km`.
2. `day.per_km = -5` — у **класі** лежить об'єкт із `__set__`, тож замість запису в екземпляр Python викликав `Positive.__set__(дескриптор, day, -5)`, і перевірка не пропустила.
3. `day.per_km` — так само викликається `__get__`.

`vars(day)` показує, де насправді лежать значення: `_base`, `_per_km`, `_min_fare`. Одне правило — три поля; новий клас з додатними полями — ще один рядок `поле = Positive()`.

### Приклад 5.2. Коли саме викликаються методи дескриптора

Дескриптор-шпигун друкує кожен свій виклик. Зверни увагу: перший друк з'явиться вже в клітинці з **визначенням** класу.

In [ ]:
class Traced:
    def __set_name__(self, owner, name):
        print(f"  __set_name__: {owner.__name__}.{name}")
        self.name = "_" + name

    def __get__(self, obj, objtype=None):
        if obj is None:
            print("  __get__ через клас")
            return self
        print(f"  __get__ {self.name}")
        return getattr(obj, self.name)

    def __set__(self, obj, value):
        print(f"  __set__ {self.name} = {value!r}")
        setattr(obj, self.name, value)


class Order:
    dish = Traced()
    qty = Traced()

    def __init__(self, dish, qty):
        self.dish = dish
        self.qty = qty

In [ ]:
print("створюю замовлення")
order = Order("Борщ", 2)
print("читаю")
print(order.dish, order.qty)
print("через клас:", type(Order.dish).__name__)

`__set_name__` спрацював **один раз на поле під час створення класу** — ще до жодного замовлення. Далі кожне присвоєння в `__init__` — це `__set__`, кожне читання — `__get__`. Звернення через клас (`Order.dish`) приходить з `obj=None`, і дескриптор повертає сам себе — саме для цього рядок `if obj is None`.

### Приклад 5.3. `property` — теж дескриптор

Порівняємо, які методи мають дескриптори з цього уроку.

In [ ]:
for name, obj in [("Positive", Positive), ("property", property), ("cached_property", cached_property)]:
    print(f"{name:16} __get__: {hasattr(obj, '__get__')}  __set__: {hasattr(obj, '__set__')}")
print(type(vars(Tariff)["base"]).__name__, type(vars(Courier)["level"]).__name__)

Дескриптори бувають двох видів:

- **data** — є `__set__`: `property`, `Positive`;
- **non-data** — лише `__get__`: звичайні методи, `cached_property`.

Від виду залежить, хто виграє, коли однойменне значення є і в класі (дескриптор), і в екземплярі:

```text
obj.attr
  1. data-дескриптор у класі?       так → його __get__   (property, Positive)
  2. attr у vars(obj)?              так → значення з екземпляра
  3. non-data дескриптор у класі?   так → його __get__   (метод, cached_property)
  4. attr у класі чи батьках (MRO)? так → атрибут класу
  5. інакше                          → AttributeError
```

### Приклад 5.4. Хто виграє: дескриптор чи екземпляр

Покладемо значення **напряму** в словник екземпляра — в обхід звичайного присвоєння.

In [ ]:
day = Tariff(40, 12, 80)
vars(day)["per_km"] = 999                  # data-дескриптор: крок 1 раніше за крок 2
print("per_km:", day.per_km, "| у словнику:", vars(day)["per_km"])

day.fare = lambda km: 0                    # метод — non-data: крок 2 раніше за крок 3
print("fare(10) з підміною:", day.fare(10))
del day.fare
print("fare(10) після del:", day.fare(10))

`per_km` лишився 12: data-дескриптор стоїть **перед** словником екземпляра, тож запис у `vars(day)` нічого не зламав. А метод `fare` — non-data, і функція в екземплярі його перекрила, поки ми її не видалили. Те саме з `cached_property` у прикладі 4.7: записавши результат у `vars(route)`, він більше не викликається.

### Приклад 5.5. Знайди помилку: дескриптор зберігає значення в собі

Дескриптор створюється **один раз на клас**, а не на кожен екземпляр. Що буде, якщо зберегти значення в самому дескрипторі?

In [ ]:
class SharedPositive:
    def __get__(self, obj, objtype=None):
        if obj is None:
            return self
        return self.value

    def __set__(self, obj, value):
        if value <= 0:
            raise ValueError("має бути більшим за 0")
        self.value = value                 # одне значення на ВСІ тарифи


class NightTariff:
    per_km = SharedPositive()

    def __init__(self, per_km):
        self.per_km = per_km

In [ ]:
early = NightTariff(12)
late = NightTariff(18)
print(early.per_km, late.per_km)

Обидва тарифи кажуть 18: `early` і `late` ділять один об'єкт `SharedPositive`, і другий тариф перезаписав значення першого. Тому `Positive` кладе значення в екземпляр — `setattr(obj, "_per_km", value)`, а сам пам'ятає лише **ім'я** поля.

### Задача 2. Дескриптор `Range`

| | `Range` |
|---|---|
| **Що це?** | правило «значення в межах `[low, high]`» для поля класу |
| **Знає** | `low`, `high` (вже в `__init__`), ім'я поля `_<ім'я>` |
| **Вміє** | `__set_name__`, `__get__` (через клас — сам дескриптор), `__set__` з перевіркою |
| **Правила** | значення поза межами → `ValueError`; значення лежить в екземплярі під `_<ім'я>` |
| **Не робить** | не знає, що таке кур'єр |

Допиши методи за зразком `Positive`. Клас `Courier` з полями `rating = Range(1, 5)` і `experience = Range(0, 50)` — у наступній клітинці, перевірка — після нього.

In [ ]:
class Range:
    def __init__(self, low, high):
        self.low = low
        self.high = high

    # BEGIN SOLUTION
    def __set_name__(self, owner, name):
        self.name = "_" + name

    def __get__(self, obj, objtype=None):
        if obj is None:
            return self
        return getattr(obj, self.name)

    def __set__(self, obj, value):
        if not self.low <= value <= self.high:
            raise ValueError(f"{self.name[1:]} має бути від {self.low} до {self.high}, а маємо {value}")
        setattr(obj, self.name, value)
    # END SOLUTION

In [ ]:
class Courier:
    rating = Range(1, 5)
    experience = Range(0, 50)

    def __init__(self, name, rating, experience):
        self.name = name
        self.rating = rating
        self.experience = experience

In [ ]:
oleh = Courier("Олег", 4.8, 3)
print(vars(oleh))
assert oleh.rating == 4.8 and isinstance(Courier.rating, Range)
assert vars(oleh) == {"name": "Олег", "_rating": 4.8, "_experience": 3}
for bad in [lambda: Courier("Ірина", 7, 2), lambda: Courier("Ірина", 4.5, -1)]:
    try:
        bad()
        raise AssertionError("мало впасти з ValueError")
    except ValueError as error:
        print(error)
couriers = [oleh, Courier("Ірина", 4.95, 5), Courier("Тарас", 4.2, 1)]
best = sorted(couriers, key=lambda courier: courier.rating, reverse=True)
assert [courier.name for courier in best] == ["Ірина", "Олег", "Тарас"]
print("✅ Задача 2 розв'язана")

---
## 6. Декоратори класів

Декоратор функції (урок 10) приймає функцію й повертає функцію. **Декоратор класу** приймає клас і повертає клас — той самий, доповнений, або новий. `@payment("card")` над класом означає `CardPayment = payment("card")(CardPayment)`.

### Приклад 6.1. Реєстр способів оплати

Кожен клас сам записується в довідник, щойно його оголосили.

In [ ]:
PAYMENTS = {}


def payment(code):
    def register(cls):
        PAYMENTS[code] = cls
        return cls
    return register


@payment("card")
class CardPayment:
    def pay(self, amount):
        return f"картка: {amount} грн"


@payment("cash")
class CashPayment:
    def pay(self, amount):
        return f"готівка кур'єру: {amount} грн"

In [ ]:
print(PAYMENTS)
for code in ["card", "cash"]:
    print(PAYMENTS[code]().pay(340))

Застосунок обирає спосіб оплати за кодом із запиту: `PAYMENTS[code]().pay(amount)`. Новий спосіб — новий клас із декоратором; жоден `if/elif` правити не треба. Це поліморфізм з уроку 21 плюс автоматична реєстрація.

### Приклад 6.2. Реєстр знижок

Той самий прийом для знижок: `discount(code)` записує клас у `DISCOUNTS[code]` і **повертає клас без змін**. Якщо забути `return cls`, ім'я класу стане `None`.

In [ ]:
DISCOUNTS = {}


def discount(code):
    def register(cls):
        DISCOUNTS[code] = cls
        return cls
    return register


@discount("student")
class StudentDiscount:
    def apply(self, amount):
        return round(amount * 0.9)


@discount("night")
class NightDiscount:
    def apply(self, amount):
        return max(amount - 50, 0)

In [ ]:
print(DISCOUNTS)
print(DISCOUNTS["student"]().apply(340), DISCOUNTS["night"]().apply(30))
assert set(DISCOUNTS) == {"student", "night"}
assert DISCOUNTS["student"]().apply(340) == 306
assert DISCOUNTS["night"]().apply(30) == 0
assert StudentDiscount.__name__ == "StudentDiscount", "декоратор має повернути клас"

Студентська знижка дає 306 із 340, нічна не опускає ціну нижче нуля. Остання перевірка показує, що після декорування `StudentDiscount` — і далі той самий клас.

### Приклад 6.3. Декоратор, що дописує метод

Декоратор може не лише зареєструвати клас, а й **доповнити** його. `auto_repr` додає `__repr__`, зібраний з атрибутів екземпляра.

In [ ]:
def auto_repr(cls):
    def __repr__(self):
        fields = ", ".join(f"{name}={value!r}" for name, value in vars(self).items())
        return f"{type(self).__name__}({fields})"
    cls.__repr__ = __repr__
    return cls


@auto_repr
class Driver:
    def __init__(self, name, car, plate):
        self.name = name
        self.car = car
        self.plate = plate

In [ ]:
print(Driver("Тарас", "Skoda", "AA1234BB"))
print([Driver("Ірина", "Kia", "KA0001AA")])

Клас `Driver` не має власного `__repr__`, але декоратор поставив його в клас, і `print` показує всі поля. Саме так, тільки набагато повніше, працює `@dataclass` нижче.

### Приклад 6.4. `@total_ordering`: решту порівнянь допише Python

Даєш `__eq__` і один із `__lt__`, `__le__`, `__gt__`, `__ge__` — декоратор дописує решту. Для грошей порядок природний, тож порівняння мають жити в класі.

In [ ]:
from functools import total_ordering


@total_ordering
class Money(Money):
    def __lt__(self, other):
        if not isinstance(other, Money):
            return NotImplemented
        return self.amount < other.amount

In [ ]:
print(Money(95) > Money(60), Money(60) <= Money(60), Money(40) >= Money(95))
print(max([Money(95), Money(40)]), sorted([Money(95), Money(40), Money(60)]))
print([name for name in ("__le__", "__gt__", "__ge__") if name in vars(Money)])

Ми написали лише `__lt__`, а у словнику класу з'явилися `__le__`, `__gt__`, `__ge__` — їх дописав декоратор, виразивши через `__lt__` і `__eq__`.

### Приклад 6.5. `@dataclass`: клас-дані без шаблонного коду

`dataclasses.dataclass` читає **анотації полів** і генерує `__init__`, `__repr__`, `__eq__`. Почнемо зі звичайного (змінюваного) датакласу.

In [ ]:
from dataclasses import dataclass, field


@dataclass
class OrderLine:
    dish: str
    qty: int = 1

In [ ]:
line = OrderLine("Борщ", 2)
print(line, line == OrderLine("Борщ", 2), OrderLine("Узвар"))
line.qty = 3
print(line)
print(OrderLine.__hash__)

`__init__` прийняв поля в порядку оголошення, `qty` має значення за замовчуванням. `repr` показує поля, `==` порівнює їх. Поле можна змінити. А `__hash__` — `None`: датаклас визначив `__eq__`, а змінюваний об'єкт хешувати небезпечно (приклад 3.3).

### Приклад 6.6. `frozen=True, order=True`

🔮 **Прогноз:** які з методів `__init__`, `__repr__`, `__eq__`, `__lt__`, `__hash__` згенерує `@dataclass(frozen=True, order=True)`?

In [ ]:
@dataclass(frozen=True, order=True)
class Money:
    amount: int

    def __add__(self, other):
        if not isinstance(other, Money):
            return NotImplemented
        return Money(self.amount + other.amount)

    def __str__(self):
        return f"{self.amount} грн"

In [ ]:
a = Money(95)
print(repr(a), a == Money(95), a < Money(100), len({a, Money(95)}))
print([name for name in ("__init__", "__repr__", "__eq__", "__lt__", "__hash__") if name in vars(Money)])
try:
    a.amount = 100
except AttributeError as error:
    print(type(error).__name__)

<details>
<summary>Відповідь</summary>

Усі п'ять. `order=True` дає порівняння `<`, `<=`, `>`, `>=`, `frozen=True` — незмінність (`FrozenInstanceError`, нащадок `AttributeError`) і, раз об'єкт незмінний, `__hash__`. Пастка з монетою, що «загубилась» у множині, тепер неможлива. Власні `__add__` і `__str__` датаклас не чіпає.

</details>

### Приклад 6.7. Знайди помилку: список за замовчуванням

Замовлення має список страв. Напишемо поле зі значенням `[]` — як звичайний аргумент за замовчуванням.

In [ ]:
try:
    @dataclass
    class Order:
        order_id: int
        dishes: list = []
except ValueError as error:
    print(error)


@dataclass
class Order:
    order_id: int
    dishes: list = field(default_factory=list)

In [ ]:
first, second = Order(1), Order(2)
first.dishes.append("Борщ")
print(first, second)

Датаклас із `dishes: list = []` навіть не створився: один спільний список на всі замовлення — та сама пастка, що й змінюваний аргумент за замовчуванням в уроці 8, і `dataclass` її забороняє. `field(default_factory=list)` викликає `list()` для **кожного** замовлення: страва потрапила лише в перше.

### Приклад 6.8. `__post_init__`: перевірка полів

Згенерований `__init__` нічого не перевіряє. Перевірку пишуть у `__post_init__` — його викликає `__init__` одразу після заповнення полів. Координати адреси доставки:

In [ ]:
@dataclass(frozen=True)
class Point:
    lat: float
    lon: float

    def __post_init__(self):
        if not -90 <= self.lat <= 90:
            raise ValueError(f"широта від -90 до 90, а маємо {self.lat}")
        if not -180 <= self.lon <= 180:
            raise ValueError(f"довгота від -180 до 180, а маємо {self.lon}")

In [ ]:
office = Point(50.45, 30.52)
print(office)
try:
    Point(150, 30)
except ValueError as error:
    print(error)

Точку з широтою 150 неможливо навіть створити — помилковий об'єкт не потрапить у систему.

### Приклад 6.9. `order=True` порівнює кортежі полів

Слоти доставки: година й хвилина. Порядок — як у кортежів: спершу перше поле, при рівності — друге. Поле, яке не має впливати на порівняння, позначають `field(compare=False)`.

In [ ]:
@dataclass(frozen=True, order=True)
class Slot:
    hour: int
    minute: int
    note: str = field(default="", compare=False)

In [ ]:
slots = [Slot(19, 30), Slot(18, 45, "біля під'їзду"), Slot(19, 0)]
print(sorted(slots))
print(Slot(18, 45) == Slot(18, 45, "інша примітка"), Slot(18, 45) < Slot(19, 0))

`sorted` упорядкував слоти як кортежі `(18, 45)`, `(19, 0)`, `(19, 30)`. Примітка не впливає ні на порядок, ні на рівність. Порядок полів у класі — це й порядок порівняння, тож переставляти поля в `order=True` датакласі небезпечно.

### Приклад 6.10. `dict`, `NamedTuple` чи `@dataclass`

Той самий запис трьома способами — і опечатка в імені поля.

In [ ]:
from typing import NamedTuple


class TripTuple(NamedTuple):
    km: float
    minutes: int


@dataclass(frozen=True)
class Trip:
    km: float
    minutes: int

In [ ]:
as_dict = {"km": 4.5, "minutes": 18}
as_tuple = TripTuple(4.5, 18)
as_class = Trip(4.5, 18)

print(as_dict.get("minuts"))                 # опечатка — тихо None
km, minutes = as_tuple                       # розпаковується як кортеж
print(km, minutes, as_tuple == (4.5, 18))
for record in [as_tuple, as_class]:
    try:
        record.minuts
    except AttributeError as error:
        print(type(record).__name__, "→", type(error).__name__)

| Варіант | Плюси | Мінуси | Коли |
|---|---|---|---|
| `dict` | нуль коду | опечатка в ключі — тихий баг (`None`); немає методів | тимчасові дані, JSON на вході |
| `NamedTuple` | незмінний, розпаковується | дорівнює звичайному кортежу `(4.5, 18)` — легко сплутати з іншим записом | прості записи, рядки з файлу |
| `@dataclass` | методи з анотацій, `frozen`, `order`, `__post_init__` | перевірки — вручну в `__post_init__` | більшість класів-даних сервісу |
| ручний клас | повний контроль | шаблонний код, легко забути `__hash__` | сутності з поведінкою й правилами |

### Задача 3. Відстань як об'єкт-значення

| | `Distance` |
|---|---|
| **Що це?** | відстань у кілометрах — об'єкт-значення |
| **Знає** | `km: float` |
| **Вміє** | `+` між відстанями; `sum(legs)` **без** стартового значення; порівняння й `max`; `str` → `"8.0 км"` |
| **Правила** | незмінна, хешується (дві однакові — один елемент множини); від'ємна відстань → `ValueError` у `__post_init__` |
| **Не робить** | не додається до чисел і грошей |

Підказка: `@dataclass(frozen=True, order=True)`, `__post_init__`, `__add__` з `NotImplemented`, `__radd__` для нуля (приклад 2.5), `__str__`. Перевірка — у наступній клітинці.

In [ ]:
# BEGIN SOLUTION
@dataclass(frozen=True, order=True)
class Distance:
    km: float

    def __post_init__(self):
        if self.km < 0:
            raise ValueError(f"відстань не може бути від'ємною: {self.km}")

    def __add__(self, other):
        if not isinstance(other, Distance):
            return NotImplemented
        return Distance(self.km + other.km)

    def __radd__(self, other):
        if other == 0:
            return self
        return NotImplemented

    def __str__(self):
        return f"{self.km} км"
# END SOLUTION

In [ ]:
legs = [Distance(2.5), Distance(4.0), Distance(1.5)]
print(sum(legs), max(legs))
assert sum(legs) == Distance(8.0) and str(sum(legs)) == "8.0 км"
assert max(legs) == Distance(4.0) and sorted(legs)[0] == Distance(1.5)
assert len({Distance(1.5), Distance(1.5)}) == 1
for bad in [lambda: Distance(-1), lambda: setattr(legs[0], "km", 10)]:
    try:
        bad()
        raise AssertionError("мало впасти")
    except (ValueError, AttributeError) as error:
        print(type(error).__name__)
print("✅ Задача 3 розв'язана")

---
## 7. Значення й сутність; розібраний приклад

| | Об'єкт-значення | Сутність |
|---|---|---|
| Приклади | `Money`, `Distance`, `Point` | `Order`, `Courier`, `Delivery` |
| Що таке «рівні» | однакові поля: `Money(95) == Money(95)` | той самий об'єкт або той самий `id` |
| Змінюваність | незмінний: «змінити» — створити новий | змінюється з часом: статус, рейтинг |
| Хеш | з полів; можна в `set` і ключем `dict` | за ідентичністю або за незмінним `id` |
| Інструмент | `@dataclass(frozen=True)` | звичайний клас, `@property`, методи-двері (урок 22) |

Який інструмент обрати:

```text
що потрібно класу?
 ├─ працювати з синтаксисом Python (len, in, +, sorted)? → dunder-методи
 ├─ поле з правилом?
 │    ├─ одне поле                         → @property + сеттер
 │    └─ те саме правило на багатьох полях → дескриптор
 ├─ клас переважно зберігає дані?          → @dataclass (frozen — для значень)
 ├─ зареєструвати чи доповнити клас?       → декоратор класу
 └─ інакше                                 → звичайний клас
```

### Приклад 7.1. Значення проти сутності

Дві однакові суми — це одне значення. Два замовлення з однаковими стравами — два **різні** замовлення. `eq=False` лишає датакласу рівність за ідентичністю й стандартний хеш.

In [ ]:
@dataclass(eq=False)
class TaxiOrder:
    order_id: int
    dish: str
    status: str = "нове"

In [ ]:
print(Money(95) == Money(95), len({Money(95), Money(95)}))

first, second = TaxiOrder(1, "Борщ"), TaxiOrder(2, "Борщ")
again = first
print(first == second, first == again, len({first, second}))
first.status = "доставлене"
print(first in {again}, again.status)

Гроші рівні за полями, і множина тримає одну суму. Замовлення рівне лише собі: два борщі — два різні замовлення. Статус змінюється, а замовлення лишається тим самим об'єктом (`again` бачить «доставлене») і знаходиться в множині, бо хеш рахується з ідентичності, а не зі змінюваних полів.

### Приклад 7.2. Кошик із грошима

Усе разом: `@dataclass` дає поля, `@property` — обчислювану суму, dunder-методи — `len` і `in`.

| | `Cart` (остаточний) |
|---|---|
| **Що це?** | кошик: список позицій `CartItem` |
| **Знає** | `items` — **свій** список для кожного кошика |
| **Вміє** | `add(dish, price, qty)`, `len` — порції, `in` — чи є страва, `for`, `total` — сума в `Money` |
| **Правила** | позицію з кількістю < 1 неможливо створити (`CartItem.__post_init__`) |
| **Не робить** | не приймає оплату (це `PAYMENTS`) і не рахує знижок (`DISCOUNTS`) |

In [ ]:
@dataclass(frozen=True, order=True)
class Money:
    amount: int

    def __add__(self, other):
        if not isinstance(other, Money):
            return NotImplemented
        return Money(self.amount + other.amount)

    def __mul__(self, times):
        return Money(self.amount * times)

    def __str__(self):
        return f"{self.amount} грн"


@dataclass(frozen=True)
class CartItem:
    dish: str
    price: Money
    qty: int = 1

    def __post_init__(self):
        if self.qty < 1:
            raise ValueError(f"кількість має бути від 1, а маємо {self.qty}")

    @property
    def total(self):
        return self.price * self.qty


@dataclass
class Cart:
    items: list = field(default_factory=list)

    def add(self, dish, price, qty=1):
        self.items.append(CartItem(dish, Money(price), qty))

    def __len__(self):
        return sum(item.qty for item in self.items)

    def __iter__(self):
        return iter(self.items)

    def __contains__(self, dish):
        return any(item.dish == dish for item in self.items)

    @property
    def total(self):
        return sum((item.total for item in self.items), Money(0))

In [ ]:
cart = Cart()
cart.add("Борщ", 95, 2)
cart.add("Узвар", 40)
print(len(cart), "Узвар" in cart, cart.total)
print(max(cart, key=lambda item: item.total))
try:
    cart.add("Вареники", 110, 0)
except ValueError as error:
    print(error)
print(len(cart))

Що тут працює:

- `Money` і `CartItem` — об'єкти-значення: `frozen=True`, рівність за полями, хеш.
- `__post_init__` не дав створити позицію з `qty=0`, тож вона не потрапила в кошик, і `len(cart)` лишився 3.
- `__mul__` дає `price * qty` → `Money`; `sum(…, Money(0))` стартує з грошей, тому `__radd__` тут не потрібен.
- `max(cart, key=…)` працює завдяки `__iter__`; друк показує `repr`, згенерований датакласом.

Поля датакласу публічні: `cart.items.append(…)` обійде перевірки `add`. Для кошика це прийнятно; де потрібен сильний інваріант — ховаємо стан за `_` і методами, як в уроці 22.

### Приклад 7.3. Об'єднати кошики

Сім'я замовляє з двох телефонів: `family = mom + son` — **новий** кошик з усіма позиціями; вихідні кошики не змінюються. З не-кошиком — `NotImplemented`.

In [ ]:
class Cart(Cart):
    def __add__(self, other):
        if not isinstance(other, Cart):
            return NotImplemented
        return Cart(self.items + other.items)

In [ ]:
mom = Cart()
mom.add("Борщ", 95, 2)
son = Cart()
son.add("Деруни", 85)
family = mom + son
print(len(family), family.total, "Деруни" in family)
print(len(mom), len(son), family is mom)
try:
    mom + 5
except TypeError as error:
    print(error)

`self.items + other.items` створює **новий** список, тож `mom` і `son` лишилися з 2 і 1 порцією. Ділити самі позиції між кошиками безпечно — `CartItem` незмінні. `mom + 5` — `TypeError`, бо `__add__` повернув `NotImplemented`.

## ✅ Самоперевірка

1. Що викликає Python для `if cart:`, якщо немає `__bool__`?
2. Навіщо `__add__` повертає `NotImplemented`?
3. Чому після `__eq__` об'єкт не лізе в `set`?
4. Чому в сеттері не можна писати `self.rating = value`?
5. Що робить `__set_name__`?
6. Чим об'єкт-значення відрізняється від сутності?
7. Чому `property` спрацьовує при кожному читанні, а `cached_property` — лише раз?
8. Чому дескриптор зберігає значення в екземплярі, а не в собі?
9. Навіщо `field(default_factory=list)` замість `= []`?

<details>
<summary>Відповіді</summary>

1. `__len__`.
2. Щоб Python спробував `__radd__` іншого операнда, а потім сам кинув `TypeError`.
3. Python прибирає `__hash__`, щоб рівні об'єкти не мали різних хешів.
4. Це знову виклик сеттера — `RecursionError`. Зберігати в `self._rating`.
5. Повідомляє дескриптору ім'я атрибута; викликається під час створення класу.
6. Значення рівні за полями й незмінні (`Money`); сутність має ідентичність і змінюється (`Order`).
7. `property` — data-дескриптор, він перед словником екземпляра; `cached_property` — non-data, після словника: записавши результат в екземпляр, він більше не викликається.
8. Дескриптор один на клас, а екземплярів багато: значення в дескрипторі стало б спільним для всіх (приклад 5.5).
9. `[]` — один список на всі об'єкти; `default_factory` створює новий для кожного (приклад 6.7).

</details>

### Шпаргалка

```python
def __len__(self): ...            # len(x), bool(x)
def __contains__(self, item): ... # item in x
def __iter__(self): ...           # for, list(x), max(x)
def __add__(self, other):         # a + b
    if not isinstance(other, Money):
        return NotImplemented
def __radd__(self, other): ...    # 0 + a  (sum)

class Positive:                   # дескриптор
    def __set_name__(self, owner, name): self.name = "_" + name
    def __get__(self, obj, objtype=None): ...
    def __set__(self, obj, value): ...

@dataclass(frozen=True, order=True)   # значення: __init__, __repr__, __eq__, <, __hash__
class Money:
    amount: int
    items: list = field(default_factory=list)   # не []!
```

## Далі

- **Практикум на реальних даних**: `lab_lesson_24_cars_descriptors.ipynb` — дескриптор `PositiveNumber` для автомобілів.
- **Урок 25 — Ітератори advanced**: власний `__next__`, `.send()`, конвеєри `itertools`.